# 01 — Data Audit

## 목적

이 Notebook은 전략 성과를 분석하거나 새로운 trading rule을 찾기 위한 Notebook이 아니다.

목표는 다음과 같다.

1. SQLite 데이터 구조 확인
2. V2.8 observation grain 및 uniqueness 검증
3. feature availability / missing semantics 확인
4. candidate cohort 정의
5. forward outcome coverage 및 integrity 검증
6. BTC / derivatives context 구조 확인
7. V2.8 shadow lane의 frozen candidate cohort 일관성 검증
8. 이후 research notebook에서 사용할 안전한 분석 모집단 정의

### 중요한 데이터 의미

- 수익률/모멘텀/MFE/MAE: `0.01 = 1%`
- Forward Outcome에는 fee/slippage가 포함되어 있지 않음
- score 계열 NULL은 오류가 아니라 scoring 대상이 아닌 market observation을 포함하기 때문
- V2.8 네 lane은 동일 candidate feature cohort를 공유
- `selected=1`은 실제 신규 ENTRY와 동일한 의미가 아님
- Squeeze/Crowding categorical state는 V2.8에서 변동 표본이 부족하므로 predictive conclusion을 내리지 않음

In [1]:
from pathlib import Path
import sqlite3
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("imports OK")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

OBS_DB = Path("data/observations/crypto-forward.sqlite3")
PAPER_DB = Path("data/paper/crypto-trading.sqlite3")

print("OBS DB  :", OBS_DB.resolve())
print("PAPER DB:", PAPER_DB.resolve())

assert OBS_DB.exists(), f"Observation DB not found: {OBS_DB}"
assert PAPER_DB.exists(), f"Paper DB not found: {PAPER_DB}"

imports OK
OBS DB  : /Users/songjaegeun/git/investment-research-engine/data/observations/crypto-forward.sqlite3
PAPER DB: /Users/songjaegeun/git/investment-research-engine/data/paper/crypto-trading.sqlite3


## 1. Read-only SQLite connection

운영 중인 DB를 분석 Notebook이 변경하지 않도록 SQLite `mode=ro`로 연결한다.

Docker가 실행 중인 환경에서는 가능한 한 동일 컨테이너 내부에서 Notebook을 실행한다.

In [2]:
def connect_readonly(path: Path):
    return sqlite3.connect(
        f"file:{path}?mode=ro",
        uri=True
    )

obs = connect_readonly(OBS_DB)
paper = connect_readonly(PAPER_DB)

print("SQLite connections opened in read-only mode.")

SQLite connections opened in read-only mode.


In [3]:
def get_tables(conn):
    return pd.read_sql_query("""
        SELECT
            name,
            type
        FROM sqlite_master
        WHERE type IN ('table', 'view')
        ORDER BY type, name
    """, conn)

obs_tables = get_tables(obs)
paper_tables = get_tables(paper)

print("=== Observation DB ===")
display(obs_tables)

print("\n=== Paper DB ===")
display(paper_tables)

=== Observation DB ===


,name,type
0,decision_market_context,table
1,decision_outcome,table
2,decision_outcome_minute,table
3,decision_selection_variant,table
4,decision_snapshot,table
5,observation_experiment,table



=== Paper DB ===


,name,type
0,paper_execution,table
1,paper_portfolio,table
2,paper_position,table
3,paper_rebalance_decision,table


In [4]:
REQUIRED_OBS_TABLES = {
    "observation_experiment",
    "decision_snapshot",
    "decision_outcome_minute",
    "decision_market_context",
}

REQUIRED_PAPER_TABLES = {
    "paper_portfolio",
    "paper_execution",
    "paper_rebalance_decision",
}

obs_table_set = set(obs_tables["name"])
paper_table_set = set(paper_tables["name"])

missing_obs = REQUIRED_OBS_TABLES - obs_table_set
missing_paper = REQUIRED_PAPER_TABLES - paper_table_set

assert not missing_obs, f"Missing observation tables: {missing_obs}"
assert not missing_paper, f"Missing paper tables: {missing_paper}"

print("Required tables: PASS")

Required tables: PASS


## 2. Experiment metadata

V2.8은 네 개의 shadow lane이 같은 시장 기간에 동시에 실행됐다.

따라서 V2.8 lane 간 비교는 이전 version 간 단순 비교보다 market-regime confounding이 작다.

반대로 V2.1 ~ V2.7은 서로 다른 기간에 실행됐으므로 단순 누적수익률 비교를 인과적으로 해석하면 안 된다.

In [5]:
experiments = pd.read_sql_query("""
    SELECT *
    FROM observation_experiment
    ORDER BY started_at
""", obs)

display(experiments)

,experiment_id,portfolio_id,strategy_version,config_hash,started_at,planned_end_at,status,starting_equity,completed_at,interruption_reason
0,paper-v2.1-forward-observation-20260814,paper-main,dynamic-intraday-v2.1,bfc471821e7df43d13061ae9d9fb3e3e205b05c69ee7eb...,2026-08-14T05:24:20.828945+00:00,2026-08-21T05:24:20.828945+00:00,INTERRUPTED,"937,501.915904",2026-08-16T04:08:49.228155+00:00,early safety failure: excessive drawdown and p...
1,paper-v2.1-decision-only-analysis-20260816,paper-analysis-main,dynamic-intraday-v2.1,bfc471821e7df43d13061ae9d9fb3e3e205b05c69ee7eb...,2026-08-16T04:09:16.151066+00:00,2026-08-23T04:09:16.151066+00:00,INTERRUPTED,"1,000,000.000000",2026-08-19T10:11:54.980147+00:00,superseded by paper-v2.2-decision-only-analysi...
2,paper-v2.2-decision-only-analysis-20260819,paper-v2.2-analysis-main,dynamic-intraday-v2.2,230ab870a2ed11541dc91fa5a622c7a0d385af30fa14e5...,2026-08-19T10:11:54.959587+00:00,2026-08-26T10:11:54.959587+00:00,INTERRUPTED,"1,000,000.000000",2026-08-21T09:56:06.848107+00:00,superseded by paper-v2.3-adaptive-turnover-202...
3,paper-v2.3-adaptive-turnover-20260821,paper-v2.3-adaptive-main,dynamic-intraday-v2.3,f277b3636910a2eae6cae1e6f343943b72755b3211f9e1...,2026-08-21T09:56:06.815242+00:00,2026-08-28T09:56:06.815242+00:00,COMPLETED,"1,000,000.000000",2026-08-28T09:56:06.815242+00:00,None
4,paper-v2.4-cost-aware-shadow-21a7c3-20260824,paper-v2.4-shadow-21a7c3-main,dynamic-intraday-v2.4,21a7c3b12192d233ef746c60fe88ed8c9aa5cbd7d25f15...,2026-08-24T02:29:50.286871+00:00,2026-08-31T02:29:50.286871+00:00,INTERRUPTED,"1,000,000.000000",2026-08-27T11:43:13.340928+00:00,superseded by paper-v2.3-adaptive-turnover-202...
5,paper-v2.5-four-hour-basis-shadow-1760aca-2026...,paper-v2.5-shadow-1760aca-main,dynamic-intraday-v2.5,1760aca5e35d45fbec2da677d2a92d060766a95610983d...,2026-08-27T11:43:13.195984+00:00,2026-09-03T11:43:13.195984+00:00,INTERRUPTED,"1,000,000.000000",2026-08-30T01:49:56.859146+00:00,superseded by paper-v2.3-adaptive-turnover-202...
6,paper-v2.6-regime-gate-shadow-c237ee8-20260830,paper-v2.6-shadow-c237ee8-main,dynamic-intraday-v2.6,c237ee87b57de82c667095676d653425672a711dd9b8a0...,2026-08-30T01:49:56.675429+00:00,2026-09-06T01:49:56.675429+00:00,INTERRUPTED,"1,000,000.000000",2026-09-01T04:56:44.613207+00:00,superseded by paper-v2.7-crowding-paper-5d0928...
7,paper-v2.7-crowding-paper-5d09282-20260901,paper-v2.7-crowding-5d09282-main,dynamic-intraday-v2.7,5d09282bcde0b7a1af73396bbc4ed8b94f3a90d5c49f7b...,2026-09-01T04:56:44.532892+00:00,2026-09-08T04:56:44.532892+00:00,INTERRUPTED,"1,000,000.000000",2026-09-01T11:41:02.285910+00:00,superseded by paper-v2.7-accuracy-fill-v2-a1-2...
8,paper-v2.7-accuracy-fill-v2-a1-20260901,paper-v2.7-accuracy-fill-v2-a1-main,dynamic-intraday-v2.7-accuracy-v1,76cf74654afbe93e0a986c4d6f7a776e8980e01c6c3ca7...,2026-09-01T11:41:02.097655+00:00,2026-09-08T11:41:02.097655+00:00,COMPLETED,"1,000,000.000000",2026-09-08T11:41:02.097655+00:00,None
9,paper-v2.8-ablation-20260909-production-control,paper-v2.8-ablation-20260909-production-contro...,dynamic-intraday-v2.8-production-control,7b02eb55a801373bba832c0f429d0e07d4381754d90ae1...,2026-09-09T11:34:46.285985+00:00,2026-09-16T11:34:46.285985+00:00,RUNNING,"1,000,000.000000",None,None


In [6]:
v28_experiments = experiments[
    experiments["strategy_version"].astype(str).str.startswith("V2.8", na=False)
].copy()

# strategy_version 표기가 다른 경우 experiment_id로 fallback
if v28_experiments.empty:
    v28_experiments = experiments[
        experiments["experiment_id"].str.contains("v2.8", case=False, na=False)
    ].copy()

display(v28_experiments)

V28_EXPERIMENT_IDS = v28_experiments["experiment_id"].tolist()

print("V2.8 experiment count:", len(V28_EXPERIMENT_IDS))
print()

for experiment_id in V28_EXPERIMENT_IDS:
    print(experiment_id)

assert V28_EXPERIMENT_IDS, "No V2.8 experiments found"

,experiment_id,portfolio_id,strategy_version,config_hash,started_at,planned_end_at,status,starting_equity,completed_at,interruption_reason
9,paper-v2.8-ablation-20260909-production-control,paper-v2.8-ablation-20260909-production-contro...,dynamic-intraday-v2.8-production-control,7b02eb55a801373bba832c0f429d0e07d4381754d90ae1...,2026-09-09T11:34:46.285985+00:00,2026-09-16T11:34:46.285985+00:00,RUNNING,"1,000,000.000000",None,None
10,paper-v2.8-ablation-20260909-momentum,paper-v2.8-ablation-20260909-momentum-paper,dynamic-intraday-v2.8-momentum,24efd2f5371a98259f4a95a3952007de50fcb9ad6f5691...,2026-09-09T11:34:46.285985+00:00,2026-09-16T11:34:46.285985+00:00,RUNNING,"1,000,000.000000",None,None
11,paper-v2.8-ablation-20260909-long-short,paper-v2.8-ablation-20260909-long-short-paper,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-09T11:34:46.285985+00:00,2026-09-16T11:34:46.285985+00:00,RUNNING,"1,000,000.000000",None,None
12,paper-v2.8-ablation-20260909-soft-penalty,paper-v2.8-ablation-20260909-soft-penalty-paper,dynamic-intraday-v2.8-soft-penalty,52fe019fe3ce8baea89cb5482e5835624ec56bbb01aa60...,2026-09-09T11:34:46.285985+00:00,2026-09-16T11:34:46.285985+00:00,RUNNING,"1,000,000.000000",None,None


V2.8 experiment count: 4

paper-v2.8-ablation-20260909-production-control
paper-v2.8-ablation-20260909-momentum
paper-v2.8-ablation-20260909-long-short
paper-v2.8-ablation-20260909-soft-penalty


## 3. V2.8 Snapshot load

`decision_snapshot`의 grain:

> 특정 experiment의 특정 decision 시점에서 특정 asset에 대한 하나의 전략 observation

가장 안전한 식별자는 `snapshot_id`.

논리적으로는 다음 조합도 현재 데이터에서 unique해야 한다.

- `(experiment_id, decision_id, asset)`

In [7]:
placeholders = ",".join(["?"] * len(V28_EXPERIMENT_IDS))

snapshots = pd.read_sql_query(
    f"""
    SELECT *
    FROM decision_snapshot
    WHERE experiment_id IN ({placeholders})
    """,
    obs,
    params=V28_EXPERIMENT_IDS,
)

print(f"V2.8 snapshot rows: {len(snapshots):,}")
print(f"Columns: {len(snapshots.columns):,}")

display(snapshots.head())

V2.8 snapshot rows: 674,838
Columns: 36


,snapshot_id,experiment_id,decision_id,strategy_version,config_hash,decision_time,asset,market,action,reason,score,rank,eligible,selected,current_position,target_position,portfolio_cash,portfolio_equity,current_exposure,target_exposure,reference_price,liquidity,hour_of_day,day_of_week,momentum_1h,momentum_4h,momentum_24h,volatility,reference_at,selected_rank,raw_score,score_penalty,expected_relative_return_1h,expected_relative_return_4h,fee_adjusted_expected_return,candidate_reasons_json
0,8ccfb8596834cad2bd93d0fef99bf467,paper-v2.8-ablation-20260909-long-short,f50b4c0f759768aa119fcd1c,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-09T11:35:04.757280+00:00,0G,0GKRW,REJECTED_ENTRY,NOT_SELECTED_BY_RANK,NaN,NaN,1,0,0.000000,0.000000,"1,000,000.000000","1,000,000.000000",0.000000,0.000000,270.000000,"25,066,430.662634",11,2,0.011236,0.003717,0.011236,0.005705,2026-09-09T11:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,[]
1,18805e00757aeb29a9cff8db65da74fd,paper-v2.8-ablation-20260909-long-short,f50b4c0f759768aa119fcd1c,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-09T11:35:04.757280+00:00,1INCH,1INCHKRW,REJECTED_ENTRY,NOT_SELECTED_BY_RANK,NaN,NaN,1,0,0.000000,0.000000,"1,000,000.000000","1,000,000.000000",0.000000,0.000000,127.000000,"3,107,680.458892",11,2,0.007937,0.000000,0.000000,0.005142,2026-09-09T11:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,[]
2,6e2b38e221497505b54eabcaec3727d0,paper-v2.8-ablation-20260909-long-short,f50b4c0f759768aa119fcd1c,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-09T11:35:04.757280+00:00,2Z,2ZKRW,REJECTED_ENTRY,NOT_SELECTED_BY_RANK,NaN,NaN,1,0,0.000000,0.000000,"1,000,000.000000","1,000,000.000000",0.000000,0.000000,69.100000,"5,313,638.824627",11,2,0.000000,-0.008608,-0.008608,0.005120,2026-09-09T11:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,[]
3,db9b32d656d044fcf6805883fb61a300,paper-v2.8-ablation-20260909-long-short,f50b4c0f759768aa119fcd1c,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-09T11:35:04.757280+00:00,AAVE,AAVEKRW,REJECTED_ENTRY,NOT_SELECTED_BY_RANK,NaN,NaN,1,0,0.000000,0.000000,"1,000,000.000000","1,000,000.000000",0.000000,0.000000,"175,200.000000","23,250,835.968008",11,2,0.002288,-0.002846,-0.006239,0.003283,2026-09-09T11:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,[]
4,f9a66f9ca9db09889433262b038d7ac1,paper-v2.8-ablation-20260909-long-short,f50b4c0f759768aa119fcd1c,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-09T11:35:04.757280+00:00,ADA,ADAKRW,REJECTED_ENTRY,NOT_SELECTED_BY_RANK,NaN,NaN,1,0,0.000000,0.000000,"1,000,000.000000","1,000,000.000000",0.000000,0.000000,297.000000,"164,431,995.315395",11,2,0.000000,-0.006689,0.006780,0.005236,2026-09-09T11:30:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,[]


In [8]:
TIME_COLUMNS = [
    "decision_time",
    "reference_at",
]

for column in TIME_COLUMNS:
    if column in snapshots.columns:
        snapshots[column] = pd.to_datetime(
            snapshots[column],
            utc=True,
            errors="coerce",
        )

snapshots[TIME_COLUMNS].head()

,decision_time,reference_at
0,2026-09-09 11:35:04.757280+00:00,2026-09-09 11:30:00+00:00
1,2026-09-09 11:35:04.757280+00:00,2026-09-09 11:30:00+00:00
2,2026-09-09 11:35:04.757280+00:00,2026-09-09 11:30:00+00:00
3,2026-09-09 11:35:04.757280+00:00,2026-09-09 11:30:00+00:00
4,2026-09-09 11:35:04.757280+00:00,2026-09-09 11:30:00+00:00


In [9]:
lane_counts = (
    snapshots.groupby("experiment_id")
    .agg(
        rows=("snapshot_id", "size"),
        decisions=("decision_id", "nunique"),
        assets=("asset", "nunique"),
        start=("decision_time", "min"),
        end=("decision_time", "max"),
    )
    .sort_index()
)

display(lane_counts)

,rows,decisions,assets,start,end
experiment_id,,,,,
paper-v2.8-ablation-20260909-long-short,168566,586,288,2026-09-09 11:35:04.757280+00:00,2026-09-15 13:50:04.470476+00:00
paper-v2.8-ablation-20260909-momentum,168566,586,288,2026-09-09 11:35:04.757280+00:00,2026-09-15 13:50:04.470476+00:00
paper-v2.8-ablation-20260909-production-control,168853,587,288,2026-09-09 11:35:04.757280+00:00,2026-09-15 13:50:04.470476+00:00
paper-v2.8-ablation-20260909-soft-penalty,168853,587,288,2026-09-09 11:35:04.757280+00:00,2026-09-15 13:50:04.470476+00:00


## 4. Observation grain / uniqueness

중복 observation이 존재하면 이후 forward outcome JOIN 및 lane 비교에서 row explosion이 발생할 수 있다.

In [10]:
snapshot_id_unique = snapshots["snapshot_id"].is_unique

logical_duplicate_count = snapshots.duplicated(
    ["experiment_id", "decision_id", "asset"]
).sum()

time_market_duplicate_count = snapshots.duplicated(
    ["experiment_id", "decision_time", "market"]
).sum()

print("snapshot_id unique:")
print(snapshot_id_unique)

print("\n(experiment_id, decision_id, asset) duplicates:")
print(logical_duplicate_count)

print("\n(experiment_id, decision_time, market) duplicates:")
print(time_market_duplicate_count)

snapshot_id unique:
True

(experiment_id, decision_id, asset) duplicates:
0

(experiment_id, decision_time, market) duplicates:
0


In [11]:
assert snapshots["snapshot_id"].is_unique
assert logical_duplicate_count == 0

print("Snapshot grain: PASS")

Snapshot grain: PASS


## 5. Temporal integrity

중요한 invariant:

`reference_at <= decision_time`

판단 시점 이후의 시장 데이터를 feature 계산에 사용하면 look-ahead bias가 발생한다.

In [12]:
reference_after_decision = snapshots[
    snapshots["reference_at"].notna()
    & snapshots["decision_time"].notna()
    & (snapshots["reference_at"] > snapshots["decision_time"])
]

print(
    "reference_at > decision_time:",
    len(reference_after_decision)
)

assert len(reference_after_decision) == 0

print("Temporal integrity: PASS")

reference_at > decision_time: 0
Temporal integrity: PASS


In [13]:
snapshots["reference_age_seconds"] = (
    snapshots["decision_time"]
    - snapshots["reference_at"]
).dt.total_seconds()

display(
    snapshots["reference_age_seconds"].describe(
        percentiles=[.01, .05, .25, .5, .75, .95, .99]
    )
)

count   642,212.000000
mean        452.255208
std         697.032989
min         303.113758
1%          303.239316
5%          303.334678
25%         303.666570
50%         303.822225
75%         304.003906
95%       1,203.901924
99%       3,003.987681
max      30,003.586425
Name: reference_age_seconds, dtype: float64

## 6. Feature availability

NULL 자체를 오류로 취급하지 않는다.

특히:

- score/raw_score/rank/expected return은 scoring candidate에서만 존재
- reference price/momentum/volatility NULL은 insufficient history, stale data, excluded market 등의 상태일 수 있음

따라서 `null rate`와 `data quality failure`를 구분한다.

In [14]:
FEATURE_COLUMNS = [
    "score",
    "raw_score",
    "score_penalty",
    "rank",
    "reference_price",
    "liquidity",
    "momentum_1h",
    "momentum_4h",
    "momentum_24h",
    "volatility",
    "expected_relative_return_1h",
    "expected_relative_return_4h",
    "fee_adjusted_expected_return",
    "current_position",
    "target_position",
    "current_exposure",
    "target_exposure",
]

FEATURE_COLUMNS = [
    c for c in FEATURE_COLUMNS
    if c in snapshots.columns
]

FEATURE_COLUMNS

['score',
 'raw_score',
 'score_penalty',
 'rank',
 'reference_price',
 'liquidity',
 'momentum_1h',
 'momentum_4h',
 'momentum_24h',
 'volatility',
 'expected_relative_return_1h',
 'expected_relative_return_4h',
 'fee_adjusted_expected_return',
 'current_position',
 'target_position',
 'current_exposure',
 'target_exposure']

In [15]:
def null_profile(df, columns):
    result = []

    for column in columns:
        s = df[column]

        result.append({
            "feature": column,
            "rows": len(s),
            "non_null": s.notna().sum(),
            "null": s.isna().sum(),
            "null_pct": s.isna().mean() * 100,
            "unique": s.nunique(dropna=True),
        })

    return pd.DataFrame(result).set_index("feature")


snapshot_null_profile = null_profile(
    snapshots,
    FEATURE_COLUMNS,
)

display(snapshot_null_profile)

,rows,non_null,null,null_pct,unique
feature,,,,,
score,674838,46920,627918,93.047220,1074
raw_score,674838,46920,627918,93.047220,1138
score_penalty,674838,46920,627918,93.047220,207
rank,674838,46920,627918,93.047220,20
reference_price,674838,585168,89670,13.287633,8908
liquidity,674838,560204,114634,16.986892,139927
momentum_1h,674838,560204,114634,16.986892,33543
momentum_4h,674838,560204,114634,16.986892,42227
momentum_24h,674838,560204,114634,16.986892,55134


In [16]:
def numeric_profile(df, columns):
    rows = []

    for column in columns:
        s = pd.to_numeric(
            df[column],
            errors="coerce"
        ).dropna()

        if s.empty:
            continue

        rows.append({
            "feature": column,
            "count": len(s),
            "min": s.min(),
            "p01": s.quantile(.01),
            "p05": s.quantile(.05),
            "p25": s.quantile(.25),
            "median": s.median(),
            "p75": s.quantile(.75),
            "p95": s.quantile(.95),
            "p99": s.quantile(.99),
            "max": s.max(),
            "mean": s.mean(),
            "std": s.std(),
            "unique": s.nunique(),
        })

    return pd.DataFrame(rows).set_index("feature")


snapshot_numeric_profile = numeric_profile(
    snapshots,
    FEATURE_COLUMNS,
)

display(snapshot_numeric_profile)

,count,min,p01,p05,p25,median,p75,p95,p99,max,mean,std,unique
feature,,,,,,,,,,,,,
score,46920,0.000000,0.028947,0.131579,0.373684,0.534211,0.647368,0.750215,0.763573,0.764285,0.497100,0.190117,1074
raw_score,46920,0.000000,0.028947,0.131579,0.373684,0.534211,0.647368,0.771053,0.828947,0.934211,0.500000,0.194414,1138
score_penalty,46920,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.019633,0.064662,0.213323,0.002900,0.012286,207
rank,46920,1.000000,1.000000,1.950000,5.750000,10.500000,15.250000,19.050000,20.000000,20.000000,10.500000,5.766343,20
reference_price,585168,0.000423,0.007100,1.280000,24.100000,100.000000,392.000000,"9,110.000000","3,380,330.000000","107,599,000.000000","462,354.497545","6,676,024.686177",8908
liquidity,560204,"910,597.658938","1,945,712.206448","2,886,361.308061","5,870,270.406285","12,160,801.919259","35,006,358.943307","265,606,214.811533","1,042,772,547.777573","3,750,774,328.266417","66,094,643.654785","212,833,288.246834",139927
momentum_1h,560204,-0.272904,-0.039409,-0.018576,-0.006297,0.000000,0.005464,0.018692,0.041667,0.478723,-0.000075,0.015666,33543
momentum_4h,560204,-0.284600,-0.070776,-0.033675,-0.011706,-0.001221,0.008511,0.035211,0.089286,0.853333,-0.000375,0.030586,42227
momentum_24h,560204,-0.485294,-0.137969,-0.075964,-0.030238,-0.005731,0.020292,0.084825,0.234533,1.808081,-0.000014,0.070663,55134


## 7. Analysis cohorts

전체 market observation과 실제 scoring candidate를 섞지 않는다.

분석 모집단을 명시적으로 분리한다.

In [17]:
market_observations = snapshots.copy()

feature_available = snapshots[
    snapshots["reference_price"].notna()
    & snapshots["momentum_1h"].notna()
].copy()

candidates = snapshots[
    snapshots["score"].notna()
].copy()

selected = candidates[
    candidates["selected"] == 1
].copy()

cohort_summary = pd.DataFrame({
    "rows": [
        len(market_observations),
        len(feature_available),
        len(candidates),
        len(selected),
    ]
}, index=[
    "market_observations",
    "feature_available",
    "scoring_candidates",
    "selected_candidates",
])

cohort_summary["pct_of_market"] = (
    cohort_summary["rows"]
    / len(market_observations)
    * 100
)

display(cohort_summary)

,rows,pct_of_market
market_observations,674838,100.000000
feature_available,560204,83.013108
scoring_candidates,46920,6.952780
selected_candidates,435,0.064460


In [18]:
if "eligible" in snapshots.columns:
    eligible_summary = (
        snapshots["eligible"]
        .value_counts(dropna=False)
        .rename_axis("eligible")
        .to_frame("rows")
    )

    eligible_summary["pct"] = (
        eligible_summary["rows"]
        / len(snapshots)
        * 100
    )

    display(eligible_summary)

,rows,pct
eligible,,
1,560204,83.013108
0,114634,16.986892


In [19]:
if "action" in snapshots.columns:
    action_summary = (
        snapshots["action"]
        .value_counts(dropna=False)
        .rename_axis("action")
        .to_frame("rows")
    )

    action_summary["pct"] = (
        action_summary["rows"]
        / len(snapshots)
        * 100
    )

    display(action_summary)

,rows,pct
action,,
REJECTED_ENTRY,559665,82.933237
NO_ACTION,114634,16.986892
HOLD,251,0.037194
EXIT,135,0.020005
ENTRY,133,0.019708
REJECTED_REPLACEMENT,18,0.002667
REPLACE,2,0.000296


### 중요

`selected == 1`은 실제 매수 체결과 동일하지 않다.

기존 포지션 유지도 selected일 수 있다.

실제 trade lifecycle 분석은 이후 Notebook에서:

- `action == ENTRY / EXIT`
- `paper_rebalance_decision`
- `paper_execution`

을 함께 사용한다.

In [20]:
def parse_reasons(value):
    if value is None or pd.isna(value):
        return []

    try:
        parsed = json.loads(value)

        if isinstance(parsed, list):
            return parsed

        return [parsed]

    except Exception:
        return ["__INVALID_JSON__"]


if "candidate_reasons_json" in snapshots.columns:

    reason_series = (
        snapshots["candidate_reasons_json"]
        .map(parse_reasons)
        .explode()
        .dropna()
    )

    reason_counts = (
        reason_series
        .value_counts()
        .rename_axis("reason")
        .to_frame("count")
    )

    display(reason_counts.head(50))

,count
reason,
NEW_ENTRY_FEE_ADJUSTED_RETURN_INSUFFICIENT,40568
NEW_ENTRY_WAITING_FOR_CONFIRMATIONS,35670
NEW_ENTRY_SCORE_BELOW_HURDLE,30111
NEW_ENTRY_BLOCKED_BY_FALLING_KNIFE_GUARD,14550
NEW_ENTRY_BLOCKED_BY_SHORT_TERM_SPIKE_GUARD,13652
NEW_ENTRY_BLOCKED_BY_VOLATILITY_GUARD,4796
NEW_ENTRY_BLOCKED_BY_4H_SPIKE_GUARD,4230
NEW_ENTRY_BLOCKED_BY_LOW_BTC_FUNDING_REGIME,4192
NEW_ENTRY_BLOCKED_BY_REENTRY_COOLDOWN,2067


## 8. Forward Outcome

Outcome grain:

`(snapshot_id, horizon_minutes)`

Forward return:

`future_close / reference_price - 1`

MFE:

`max(future candle high) / reference_price - 1`

MAE:

`min(future candle low) / reference_price - 1`

Forward Outcome 자체에는 fee/slippage가 포함되지 않는다.

In [21]:
v28_snapshot_ids = snapshots[
    ["snapshot_id"]
].drop_duplicates()

obs.execute("""
    DROP TABLE IF EXISTS temp.v28_snapshot_ids
""")

obs.execute("""
    CREATE TEMP TABLE v28_snapshot_ids (
        snapshot_id TEXT PRIMARY KEY
    )
""")

obs.executemany(
    """
    INSERT INTO v28_snapshot_ids(snapshot_id)
    VALUES (?)
    """,
    v28_snapshot_ids.itertuples(index=False, name=None)
)

obs.commit()

count = obs.execute("""
    SELECT COUNT(*)
    FROM v28_snapshot_ids
""").fetchone()[0]

print(f"Temp V2.8 snapshot IDs: {count:,}")

Temp V2.8 snapshot IDs: 674,838


In [23]:
outcome_coverage = pd.read_sql_query("""
    SELECT
        o.horizon_minutes,
        COUNT(*) AS outcome_rows,

        SUM(
            CASE
                WHEN o.status = 'COMPLETED'
                THEN 1 ELSE 0
            END
        ) AS completed,

        SUM(
            CASE
                WHEN o.status = 'MISSING_DATA'
                THEN 1 ELSE 0
            END
        ) AS missing_data

    FROM decision_outcome_minute o

    INNER JOIN v28_snapshot_ids v
        ON v.snapshot_id = o.snapshot_id

    GROUP BY o.horizon_minutes
    ORDER BY o.horizon_minutes
""", obs)

outcome_coverage["missing_pct"] = (
    outcome_coverage["missing_data"]
    / outcome_coverage["outcome_rows"]
    * 100
)

display(outcome_coverage)

,horizon_minutes,outcome_rows,completed,missing_data,missing_pct
0,15,584042,548154,35888,6.144764
1,30,583072,572224,10848,1.860491
2,60,581516,569452,12064,2.074577
3,240,569140,554824,14316,2.515374
4,720,535472,521472,14000,2.614516
5,1440,487308,475176,12132,2.489596


In [24]:
outcome_duplicates = pd.read_sql_query("""
    SELECT
        o.snapshot_id,
        o.horizon_minutes,
        COUNT(*) AS n

    FROM decision_outcome_minute o

    INNER JOIN v28_snapshot_ids v
        ON v.snapshot_id = o.snapshot_id

    GROUP BY
        o.snapshot_id,
        o.horizon_minutes

    HAVING COUNT(*) > 1

    LIMIT 100
""", obs)

print("Duplicate outcome PK:", len(outcome_duplicates))

display(outcome_duplicates.head())

assert outcome_duplicates.empty

print("Outcome PK integrity: PASS")

Duplicate outcome PK: 0


,snapshot_id,horizon_minutes,n


Outcome PK integrity: PASS


In [25]:
outcome_integrity = pd.read_sql_query("""
    SELECT

        SUM(
            CASE
                WHEN status = 'COMPLETED'
                 AND (
                    forward_return IS NULL
                    OR mfe IS NULL
                    OR mae IS NULL
                 )
                THEN 1 ELSE 0
            END
        ) AS completed_with_null,

        SUM(
            CASE
                WHEN status = 'MISSING_DATA'
                 AND (
                    forward_return IS NOT NULL
                    OR mfe IS NOT NULL
                    OR mae IS NOT NULL
                 )
                THEN 1 ELSE 0
            END
        ) AS missing_with_values

    FROM decision_outcome_minute o

    INNER JOIN v28_snapshot_ids v
        ON v.snapshot_id = o.snapshot_id
""", obs)

display(outcome_integrity)

,completed_with_null,missing_with_values
0,0,0


In [26]:
outcomes_4h = pd.read_sql_query("""
    SELECT
        o.snapshot_id,
        o.horizon_minutes,
        o.target_at,
        o.evaluated_at,
        o.status,
        o.forward_return,
        o.mfe,
        o.mae,
        o.missing_reason

    FROM decision_outcome_minute o

    INNER JOIN v28_snapshot_ids v
        ON v.snapshot_id = o.snapshot_id

    WHERE o.horizon_minutes = 240
""", obs)

for column in ["target_at", "evaluated_at"]:
    outcomes_4h[column] = pd.to_datetime(
        outcomes_4h[column],
        utc=True,
        errors="coerce"
    )

print(f"4h outcome rows: {len(outcomes_4h):,}")

display(outcomes_4h.head())

4h outcome rows: 569,140


,snapshot_id,horizon_minutes,target_at,evaluated_at,status,forward_return,mfe,mae,missing_reason
0,f570fa99722983386d72174fb65042af,240,2026-09-09 15:35:04.757280+00:00,2026-09-09 15:40:18.251848+00:00,COMPLETED,-0.003704,0.018519,-0.007407,None
1,8dd8f846d9dabfb41950752c55de8b4a,240,2026-09-09 15:35:04.757280+00:00,2026-09-09 15:40:18.251848+00:00,COMPLETED,-0.015748,0.000000,-0.023622,None
2,5a798e8c4d33f3c509198309775cd689,240,2026-09-09 15:35:04.757280+00:00,2026-09-09 15:40:18.251848+00:00,COMPLETED,-0.014472,0.002894,-0.020260,None
3,d3576704e8b06c56c1e8868c22d3c481,240,2026-09-09 15:35:04.757280+00:00,2026-09-09 15:40:18.251848+00:00,COMPLETED,-0.007420,0.010274,-0.013699,None
4,8fdbd0e4c76aad875d0197f6e532b7ca,240,2026-09-09 15:35:04.757280+00:00,2026-09-09 15:40:18.251848+00:00,COMPLETED,-0.016835,0.010101,-0.023569,None


In [28]:
completed_4h = outcomes_4h[
    outcomes_4h["status"] == "COMPLETED"
].copy()

display(
    completed_4h[
        ["forward_return", "mfe", "mae"]
    ].describe(
        percentiles=[
            .01, .05, .25, .5,
            .75, .95, .99
        ]
    ).T
)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
forward_return,"554,824.000000",-0.000386,0.030333,-0.454229,-0.071166,-0.033564,-0.011257,-0.000903,0.008206,0.034615,0.087432,0.853333
mfe,"554,824.000000",0.019140,0.036732,-0.018868,-0.003717,0.000000,0.004047,0.009709,0.020882,0.064302,0.170833,1.114029
mae,"554,824.000000",-0.016666,0.019050,-0.462931,-0.094726,-0.048433,-0.021492,-0.011834,-0.005714,0.000000,0.003279,0.024390


In [29]:
mfe_lt_mae = (
    completed_4h["mfe"]
    < completed_4h["mae"]
).sum()

extreme_return = (
    completed_4h["forward_return"]
    .abs()
    .gt(0.30)
    .sum()
)

print("MFE < MAE:", mfe_lt_mae)
print("|4h return| > 30%:", extreme_return)

assert mfe_lt_mae == 0

MFE < MAE: 0
|4h return| > 30%: 596


In [30]:
v28_4h = snapshots.merge(
    outcomes_4h,
    on="snapshot_id",
    how="left",
    validate="one_to_one",
)

print("Snapshot rows :", len(snapshots))
print("Merged rows   :", len(v28_4h))

assert len(v28_4h) == len(snapshots)

print("Snapshot → 4h Outcome join: PASS")

Snapshot rows : 674838
Merged rows   : 674838
Snapshot → 4h Outcome join: PASS


## 9. Market Context

`decision_market_context`는 종목별 데이터가 아니라 decision cycle 전체에 적용되는 BTC / derivatives context이다.

Grain:

`(experiment_id, decision_id)`

Snapshot과 연결할 경우 N:1 관계가 된다.

In [31]:
context = pd.read_sql_query(
    f"""
    SELECT *
    FROM decision_market_context
    WHERE experiment_id IN ({placeholders})
    """,
    obs,
    params=V28_EXPERIMENT_IDS,
)

print(f"Context rows: {len(context):,}")

display(context.head())

Context rows: 2,346


,experiment_id,decision_id,strategy_version,config_hash,decision_time,context_json
0,paper-v2.8-ablation-20260909-long-short,001050a6955845b4a9e0a3d0,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-12T05:50:03.922816+00:00,"{""ageSeconds"":119.876254,""availability"":""AVAIL..."
1,paper-v2.8-ablation-20260909-long-short,00c425fa641052493570d84e,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-14T17:20:04.087044+00:00,"{""ageSeconds"":119.956844,""availability"":""AVAIL..."
2,paper-v2.8-ablation-20260909-long-short,00f59656a3b8f8acb6d08470,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-11T10:35:03.846953+00:00,"{""ageSeconds"":119.820885,""availability"":""AVAIL..."
3,paper-v2.8-ablation-20260909-long-short,0140321bdb8860930e12a746,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-11T02:50:03.941244+00:00,"{""ageSeconds"":119.944758,""availability"":""AVAIL..."
4,paper-v2.8-ablation-20260909-long-short,01437a49fec628a9ab10e4e8,dynamic-intraday-v2.8-long-short,e6322cc3fb19946d3444343e7112989cce592772d5bd24...,2026-09-13T17:35:03.423941+00:00,"{""ageSeconds"":119.614105,""availability"":""AVAIL..."


In [32]:
context_duplicate_count = context.duplicated(
    ["experiment_id", "decision_id"]
).sum()

print("Context logical duplicates:", context_duplicate_count)

assert context_duplicate_count == 0

Context logical duplicates: 0


In [34]:
def safe_json_load(value):
    try:
        return json.loads(value)
    except Exception:
        return None


context["_parsed"] = context[
    "context_json"
].map(safe_json_load)

invalid_context_json = context["_parsed"].isna().sum()

print("Invalid context JSON:", invalid_context_json)

assert invalid_context_json == 0

context_flat = pd.json_normalize(
    context["_parsed"]
)

context_keys = sorted(context_flat.columns)

print("Context JSON paths:", len(context_keys))

context_keys

Invalid context JSON: 0
Context JSON paths: 67


['ageSeconds',
 'availability',
 'basisInputRate',
 'basisInputSource',
 'basisRate',
 'bearishUnwindScore',
 'bullishUnwindScore',
 'coinbaseObservedAt',
 'coinbasePremiumRate',
 'coinbasePriceUsd',
 'crowdingAgeSeconds',
 'crowdingAvailability',
 'crowdingConfidence',
 'crowdingDominantSide',
 'crowdingFeatureVersion',
 'crowdingIntensity',
 'crowdingSignalAsOf',
 'crowdingSnapshotId',
 'crowdingState',
 'decisionAsOf',
 'decisionDiagnostics.candidateConfirmationMeaning',
 'decisionDiagnostics.crowdingGateDisagreement',
 'decisionDiagnostics.currentCrowdingGate.blocked',
 'decisionDiagnostics.currentCrowdingGate.reason',
 'decisionDiagnostics.derivativesCostPenalty',
 'decisionDiagnostics.derivativesOverlayMode',
 'decisionDiagnostics.entryEligibleConfirmationMeaning',
 'decisionDiagnostics.expectedReturnCalibrationIsEmpirical',
 'decisionDiagnostics.policyCrowdingGate.blocked',
 'decisionDiagnostics.policyCrowdingGate.minimumBearishUnwindScore',
 'decisionDiagnostics.policyCrowdingG

In [37]:
NUMERIC_CONTEXT_COLUMNS = [
    "basisInputRate",
    "markIndexBasisRate",
    "fundingRate",
    "openInterestUsd",
    "openInterestChange15m",
    "openInterestChange1h",
    "globalLongShortRatio",
    "topPositionLongShortRatio",
    "coinbasePremiumRate",
    "coinbasePriceUsd",
    "spotReturn15m",
    "spotVolumeRatio",
    "futuresPriceChange15m",
    "futuresPriceChange1h",
    "crowdingIntensity",
    "longCrowdingScore",
    "shortCrowdingScore",
    "bullishUnwindScore",
    "bearishUnwindScore",
    "longLiquidationUsd15m",
    "shortLiquidationUsd15m",
    "ageSeconds",
]

for column in NUMERIC_CONTEXT_COLUMNS:
    if column in context_flat.columns:
        context_flat[column] = pd.to_numeric(
            context_flat[column],
            errors="coerce",
        )

In [38]:
context_meta_columns = [
    c for c in [
        "experiment_id",
        "decision_id",
        "decision_time",
    ]
    if c in context.columns
]

context_v28 = pd.concat(
    [
        context[
            context_meta_columns
        ].reset_index(drop=True),

        context_flat.reset_index(drop=True),
    ],
    axis=1,
)

if "decision_time" in context_v28.columns:
    context_v28["decision_time"] = pd.to_datetime(
        context_v28["decision_time"],
        utc=True,
        errors="coerce",
    )

display(context_v28.head())

,experiment_id,decision_id,decision_time,ageSeconds,availability,basisInputRate,basisInputSource,basisRate,bearishUnwindScore,bullishUnwindScore,coinbaseObservedAt,coinbasePremiumRate,coinbasePriceUsd,crowdingAgeSeconds,crowdingAvailability,crowdingConfidence,crowdingDominantSide,crowdingFeatureVersion,crowdingIntensity,crowdingSignalAsOf,crowdingSnapshotId,crowdingState,decisionAsOf,executionModelVersion,featureVersion,feeRate,fuelScore,fundingRate,futuresPriceChange15m,futuresPriceChange1h,globalLongShortRatio,ignitionScore,indexPrice,liquidationConfirmed,longCrowdingScore,longLiquidationUsd15m,markIndexBasisRate,markPrice,missingFields,openInterestChange15m,openInterestChange1h,openInterestUsd,reasonCodes,recommendation,shortCrowdingScore,shortLiquidationUsd15m,signalAsOf,slippageRate,snapshotAvailableAt,snapshotId,spotBreakout,spotReturn15m,spotVolumeRatio,state,symbol,takerBuySellRatio,topPositionLongShortRatio,decisionDiagnostics.candidateConfirmationMeaning,decisionDiagnostics.crowdingGateDisagreement,decisionDiagnostics.currentCrowdingGate.blocked,decisionDiagnostics.currentCrowdingGate.reason,decisionDiagnostics.derivativesCostPenalty,decisionDiagnostics.derivativesOverlayMode,decisionDiagnostics.entryEligibleConfirmationMeaning,decisionDiagnostics.expectedReturnCalibrationIsEmpirical,decisionDiagnostics.policyCrowdingGate.blocked,decisionDiagnostics.policyCrowdingGate.minimumBearishUnwindScore,decisionDiagnostics.policyCrowdingGate.minimumLongCrowdingScore,decisionDiagnostics.rawDerivativesGateRole,decisionDiagnostics.squeezeRole
0,paper-v2.8-ablation-20260909-long-short,001050a6955845b4a9e0a3d0,2026-09-12 05:50:03.922816+00:00,119.876254,AVAILABLE,-0.000490,MARK_INDEX_PROXY,None,0.018581,0.201007,2026-09-12T05:48:03.810089+00:00,-0.000186,"77,223.800000",119.876254,AVAILABLE,1.000000,NONE,btc-crowding-v1,0.000000,2026-09-12T05:48:04.046562+00:00,a64204e0eb4deed6ba7e01667173881077941e752ca3b7...,NEUTRAL,2026-09-12T05:50:03.922816+00:00,paper-fill-v2,btc-squeeze-v3-mark-index-basis-proxy,0.0005,0.000000,0.000039,0.000034,-0.000140,1.603500,0.164868,77238.15173913,True,0.000000,0.000000,-0.000490,77200.28624638,[basis_rate],0.000279,-0.000100,"7,995,733,698.108552","[NO_DERIVATIVES_RISK_ON_CONFIRMATION, NO_SQUEE...",NO_CONFIRMATION,0.000000,0.000000,2026-09-12T05:48:04.046562+00:00,0.0005,2026-09-12T05:48:04.046562+00:00,a64204e0eb4deed6ba7e01667173881077941e752ca3b7...,False,0.000743,0.811055,NONE,BTCUSDT,2.4258,2.203400,CONSECUTIVE_SCORE_ABOVE_HURDLE,False,False,None,0.000000,LONG_SHORT_GATE,SHADOW_SCORE_GUARDS_COST_DERIVATIVES_AND_CROWDING,False,False,NaN,NaN,FUNDING_BASIS_GLOBAL_LONG_SHORT_RISK_ON_GATE,BTC_DERIVATIVES_STRUCTURE_DIAGNOSTIC_AND_POINT...
1,paper-v2.8-ablation-20260909-long-short,00c425fa641052493570d84e,2026-09-14 17:20:04.087044+00:00,119.956844,AVAILABLE,-0.000339,MARK_INDEX_PROXY,None,0.239008,0.097666,2026-09-14T17:18:01.688394+00:00,-0.000190,"78,850.460000",119.956844,AVAILABLE,1.000000,NONE,btc-crowding-v1,0.000000,2026-09-14T17:18:04.130200+00:00,fcd5c18b46b48b4f79f8f90af600ff6cb2d634059f2164...,NEUTRAL,2026-09-14T17:20:04.087044+00:00,paper-fill-v2,btc-squeeze-v3-mark-index-basis-proxy,0.0005,0.000000,0.000033,0.000890,0.002717,1.256800,0.001125,78865.40782609,True,0.000000,0.000000,-0.000339,78838.69826812,[basis_rate],-0.001604,-0.000870,"8,314,017,706.497776","[NO_DERIVATIVES_RISK_ON_CONFIRMATION, NO_SQUEE...",NO_CONFIRMATION,0.000000,236.463100,2026-09-14T17:18:04.130200+00:00,0.0005,2026-09-14T17:18:04.130200+00:00,fcd5c18b46b48b4f79f8f90af600ff6cb2d634059f2164...,False,0.000056,0.308338,NONE,BTCUSDT,0.3823,1.979900,CONSECUTIVE_SCORE_ABOVE_HURDLE,False,False,None,0.000000,LONG_SHORT_GATE,SHADOW_SCORE_GUARDS_COST_DERIVATIVES_AND_CROWDING,False,False,NaN,NaN,FUNDING_BASIS_GLOBAL_LONG_SHORT_RISK_ON_GATE,BTC_DERIVATIVES_STRUCTURE_DIAGNOSTIC_AND_POINT...
2,paper-v2.8-ablation-20260909-long-short,00f59656a3b8f8acb6d08470,2026-09-11 10:35:03.846953+00:00,119.820885,AVAILABLE,-0.000539,MARK_INDE

In [44]:
def safe_nunique(series):
    """
    list/dict/set 같은 unhashable JSON 값을
    비교 가능한 문자열로 변환한 뒤 unique count를 계산한다.
    """
    def make_hashable(value):
        if isinstance(value, (list, dict)):
            return json.dumps(
                value,
                sort_keys=True,
                ensure_ascii=False
            )

        if isinstance(value, set):
            return json.dumps(
                sorted(value),
                ensure_ascii=False
            )

        return value

    return series.map(make_hashable).nunique(dropna=True)


context_coverage = pd.DataFrame({
    "non_null": context_flat.notna().sum(),
    "null": context_flat.isna().sum(),

    "coverage_pct": (
        context_flat.notna().mean() * 100
    ),

    "unique": pd.Series({
        column: safe_nunique(context_flat[column])
        for column in context_flat.columns
    }),
}).sort_values(
    ["coverage_pct", "unique"],
    ascending=[True, True]
)

display(context_coverage)

,non_null,null,coverage_pct,unique
basisRate,0,2346,0.000000,0
decisionDiagnostics.currentCrowdingGate.reason,0,2346,0.000000,0
decisionDiagnostics.policyCrowdingGate.minimumBearishUnwindScore,587,1759,25.021313,1
decisionDiagnostics.policyCrowdingGate.minimumLongCrowdingScore,587,1759,25.021313,1
shortLiquidationUsd15m,2334,12,99.488491,315
longLiquidationUsd15m,2334,12,99.488491,388
availability,2346,0,100.000000,1
basisInputSource,2346,0,100.000000,1
crowdingAvailability,2346,0,100.000000,1
crowdingFeatureVersion,2346,0,100.000000,1


In [45]:
DERIVATIVE_COLUMNS = [
    "basisRate",
    "basisInputRate",
    "markIndexBasisRate",
    "fundingRate",
    "openInterestUsd",
    "openInterestChange15m",
    "openInterestChange1h",
    "globalLongShortRatio",
    "topPositionLongShortRatio",
    "longLiquidationUsd15m",
    "shortLiquidationUsd15m",
    "coinbasePremiumRate",
    "spotReturn15m",
    "futuresPriceChange15m",
    "futuresPriceChange1h",
    "crowdingIntensity",
    "longCrowdingScore",
    "shortCrowdingScore",
    "bullishUnwindScore",
    "bearishUnwindScore",
]

available_derivatives = [
    c for c in DERIVATIVE_COLUMNS
    if c in context_flat.columns
]

display(
    numeric_profile(
        context_flat,
        available_derivatives
    )
)

,count,min,p01,p05,p25,median,p75,p95,p99,max,mean,std,unique
feature,,,,,,,,,,,,,
basisInputRate,2346,-0.000871,-0.000655,-0.000592,-0.000495,-0.000417,-0.000351,-0.000234,-0.000128,0.000026,-0.000419,0.000112,586
markIndexBasisRate,2346,-0.000871,-0.000655,-0.000592,-0.000495,-0.000417,-0.000351,-0.000234,-0.000128,0.000026,-0.000419,0.000112,586
fundingRate,2346,0.000008,0.000012,0.000027,0.000042,0.000060,0.000072,0.000096,0.000100,0.000100,0.000059,0.000021,538
openInterestUsd,2346,"7,949,708,363.763600","7,969,301,646.636974","7,979,982,509.699870","8,023,480,467.357600","8,169,950,964.490600","8,251,782,575.918779","8,314,017,706.497776","8,362,622,421.762706","8,389,122,569.480400","8,144,169,226.488777","120,548,864.929260",586
openInterestChange15m,2346,-0.018801,-0.006792,-0.004248,-0.001037,0.000041,0.001082,0.003876,0.007378,0.012940,-0.000000,0.002587,586
openInterestChange1h,2346,-0.023171,-0.014486,-0.007295,-0.001869,0.000105,0.001717,0.006149,0.012783,0.023700,-0.000149,0.004426,586
globalLongShortRatio,2346,1.103500,1.141300,1.163100,1.449800,1.590700,1.637100,1.666700,1.683100,1.710000,1.508468,0.164576,348
topPositionLongShortRatio,2346,1.962300,1.973180,2.029500,2.124600,2.196900,2.227300,2.318525,2.420200,2.484100,2.183214,0.082524,509
longLiquidationUsd15m,2334,0.000000,0.000000,0.000000,0.000000,928.150800,"17,533.629088","201,659.488200","1,009,630.523182","2,874,081.316250","52,008.157103","209,269.116805",388


In [46]:
CATEGORICAL_CONTEXT_COLUMNS = [
    "availability",
    "crowdingAvailability",
    "crowdingState",
    "recommendation",
    "state",
    "liquidationConfirmed",
]

for column in CATEGORICAL_CONTEXT_COLUMNS:

    if column not in context_flat.columns:
        continue

    print(f"\n=== {column} ===")

    display(
        context_flat[column]
        .value_counts(dropna=False)
        .to_frame("count")
    )


=== availability ===


,count
availability,
AVAILABLE,2346



=== crowdingAvailability ===


,count
crowdingAvailability,
AVAILABLE,2346



=== crowdingState ===


...

,count
recommendation,
NO_CONFIRMATION,2346



=== state ===


,count
state,
NONE,2346



=== liquidationConfirmed ===


,count
liquidationConfirmed,
True,2334
False,12


### Derivatives 해석 주의사항

V2.8에서는:

- `basisRate`는 path는 존재하지만 실제 값은 NULL
- 분석용 basis는 `basisInputRate` 또는 `markIndexBasisRate`
- Squeeze `state`의 변동이 부족
- `recommendation`도 충분히 발화하지 않음
- `crowdingState` 역시 대부분/전부 NEUTRAL

따라서 이 Notebook에서는 존재 여부와 데이터 품질만 확인한다.

향후 predictive research에서는 categorical Squeeze/Crowding state보다 funding/OI/basis/long-short/premium 등의 continuous raw feature를 우선 탐색할 수 있다.

## 10. Frozen candidate cohort consistency

V2.8 네 lane은 같은 candidate cohort를 shadow execution하는 구조다.

따라서 다음 candidate feature는 같은 decision/asset에서 lane 간 동일해야 한다.

이 검증이 실패하면 V2.8 lane 비교에 문제가 생길 수 있다.

In [47]:
FROZEN_FEATURES = [
    "score",
    "raw_score",
    "score_penalty",
    "rank",
    "reference_price",
    "liquidity",
    "momentum_1h",
    "momentum_4h",
    "momentum_24h",
    "volatility",
    "expected_relative_return_1h",
    "expected_relative_return_4h",
    "fee_adjusted_expected_return",
]

FROZEN_FEATURES = [
    c for c in FROZEN_FEATURES
    if c in snapshots.columns
]

In [48]:
frozen_consistency = []

grouped = snapshots.groupby(
    ["decision_time", "asset"],
    sort=False
)

for feature in FROZEN_FEATURES:

    nunique = grouped[feature].nunique(
        dropna=False
    )

    mismatches = (nunique > 1).sum()

    frozen_consistency.append({
        "feature": feature,
        "groups": len(nunique),
        "mismatched_groups": mismatches,
        "consistent_pct": (
            1 - mismatches / len(nunique)
        ) * 100,
    })

frozen_consistency = pd.DataFrame(
    frozen_consistency
).set_index("feature")

display(frozen_consistency)

,groups,mismatched_groups,consistent_pct
feature,,,
score,168854,0,100.000000
raw_score,168854,0,100.000000
score_penalty,168854,0,100.000000
rank,168854,0,100.000000
reference_price,168854,0,100.000000
liquidity,168854,0,100.000000
momentum_1h,168854,0,100.000000
momentum_4h,168854,0,100.000000
momentum_24h,168854,0,100.000000


In [49]:
frozen_feature_pass = (
    frozen_consistency[
        "mismatched_groups"
    ] == 0
).all()

print(
    "Frozen feature consistency:",
    "PASS" if frozen_feature_pass else "FAIL"
)

Frozen feature consistency: FAIL


## 11. Lane-specific decision behavior

Feature는 동일하더라도 다음은 lane별로 달라질 수 있다.

- selected
- target_position
- target_exposure
- action

여기서는 성과를 분석하지 않고 단순히 decision behavior의 차이가 실제 존재하는지만 확인한다.

In [50]:
lane_behavior = (
    snapshots.groupby("experiment_id")
    .agg(
        observations=("snapshot_id", "size"),

        candidates=(
            "score",
            lambda x: x.notna().sum()
        ),

        selected=(
            "selected",
            lambda x: (x == 1).sum()
        ),

        target_positions=(
            "target_position",
            lambda x: (x > 0).sum()
        ),

        mean_target_exposure=(
            "target_exposure",
            "mean"
        ),
    )
)

lane_behavior["candidate_rate_pct"] = (
    lane_behavior["candidates"]
    / lane_behavior["observations"]
    * 100
)

lane_behavior["selected_rate_pct"] = (
    lane_behavior["selected"]
    / lane_behavior["observations"]
    * 100
)

display(lane_behavior)

,observations,candidates,selected,target_positions,mean_target_exposure,candidate_rate_pct,selected_rate_pct
experiment_id,,,,,,,
paper-v2.8-ablation-20260909-long-short,168566,11720,120,120,0.051197,6.952766,0.071189
paper-v2.8-ablation-20260909-momentum,168566,11720,120,120,0.051197,6.952766,0.071189
paper-v2.8-ablation-20260909-production-control,168853,11740,97,97,0.041320,6.952793,0.057446
paper-v2.8-ablation-20260909-soft-penalty,168853,11740,98,98,0.041742,6.952793,0.058039


In [51]:
if "action" in snapshots.columns:

    lane_action = pd.crosstab(
        snapshots["experiment_id"],
        snapshots["action"],
    )

    display(lane_action)

action,ENTRY,EXIT,HOLD,NO_ACTION,REJECTED_ENTRY,REJECTED_REPLACEMENT,REPLACE
experiment_id,,,,,,,
paper-v2.8-ablation-20260909-long-short,36,37,64,28641,139782,5,1
paper-v2.8-ablation-20260909-momentum,36,37,64,28641,139782,5,1
paper-v2.8-ablation-20260909-production-control,29,29,61,28676,140053,5,0
paper-v2.8-ablation-20260909-soft-penalty,32,32,62,28676,140048,3,0


## 12. Paper execution basic audit

실제 trading performance 분석은 여기서 하지 않는다.

이 단계에서는:

- execution 존재 여부
- 시간 범위
- experiment/portfolio 연결 가능성
- fee/slippage 데이터 품질

만 확인한다.

In [52]:
executions = pd.read_sql_query("""
    SELECT *
    FROM paper_execution
    ORDER BY executed_at
""", paper)

print(f"Execution rows: {len(executions):,}")

display(executions.head())

Execution rows: 593


,order_id,intent_id,portfolio_id,pair,side,quantity,price,fee,realized_pnl,executed_at,execution_model_version,fee_rate,slippage_rate
0,paper:dynamic-rebalance-ec71624c96da60eb1e4c,dynamic-rebalance-ec71624c96da60eb1e4c,paper-main,USDTKRW,BUY,283.2861189801699716713881020,1412.0,200.0000000000000000000000000,0,2026-08-13T16:27:23.692557+00:00,paper-fill-v1,0.0005,0
1,paper:dynamic-rebalance-ffef3a7b0bb8431de75f,dynamic-rebalance-ffef3a7b0bb8431de75f,paper-main,WLDKRW,BUY,819.6721311475409836065573770,488.0,200.0000000000000000000000000,0,2026-08-13T16:27:23.693548+00:00,paper-fill-v1,0.0005,0
2,paper:dynamic-rebalance-5932dfcb3efdd3cd1f56,dynamic-rebalance-5932dfcb3efdd3cd1f56,paper-main,USDTKRW,SELL,283.2861189801699716713881020,1413.0,200.1416430594900849858356940,-116.8555240793201133144475920,2026-08-13T16:32:00.137750+00:00,paper-fill-v1,0.0005,0
3,paper:dynamic-rebalance-9cd0bc2b6ae7f7ee69ee,dynamic-rebalance-9cd0bc2b6ae7f7ee69ee,paper-main,WLDKRW,SELL,203.4721670384761144680028872,486.0,49.44373659034969581572470159,-506.0352794246900966819231805,2026-08-13T16:32:00.138502+00:00,paper-fill-v1,0.0005,0
4,paper:dynamic-rebalance-089f6db28c17447b44a2,dynamic-rebalance-089f6db28c17447b44a2,paper-main,HOMEKRW,BUY,19074.72500363092524849283325,15.7,149.7365912785027632006687410,0,2026-08-13T16:32:00.139004+00:00,paper-fill-v1,0.0005,0


In [53]:
if "executed_at" in executions.columns:
    executions["executed_at"] = pd.to_datetime(
        executions["executed_at"],
        utc=True,
        errors="coerce"
    )

print(
    "Execution period:",
    executions["executed_at"].min(),
    "→",
    executions["executed_at"].max()
)

Execution period: 2026-08-13 16:27:23.692557+00:00 → 2026-09-15 13:05:30.144698+00:00


In [54]:
EXECUTION_NUMERIC_COLUMNS = [
    "quantity",
    "price",
    "fee",
    "fee_rate",
    "slippage_rate",
    "realized_pnl",
]

for column in EXECUTION_NUMERIC_COLUMNS:

    if column in executions.columns:

        executions[column] = pd.to_numeric(
            executions[column],
            errors="coerce",
        )

display(
    numeric_profile(
        executions,
        [
            c for c in EXECUTION_NUMERIC_COLUMNS
            if c in executions.columns
        ]
    )
)

,count,min,p01,p05,p25,median,p75,p95,p99,max,mean,std,unique
feature,,,,,,,,,,,,,
quantity,593,0.002231,0.002249,0.002322,0.002375,1.824588,445.591956,"3,696.588351","20,009.603716","33,282,981.004195","113,216.106334","1,931,216.029516",255
price,593,0.007470,14.130400,74.120000,630.000000,"135,967.950000","104,526,710.500000","107,503,486.600000","109,153,655.760000","109,840,000.000000","28,106,758.537267","46,165,231.727032",389
fee,593,9.750868,120.348977,121.925038,123.511280,124.305668,125.062500,141.867271,149.736591,200.141643,126.095424,12.315178,445
fee_rate,593,0.000500,0.000500,0.000500,0.000500,0.000500,0.000500,0.000500,0.000500,0.000500,0.000500,0.000000,1
slippage_rate,593,0.000000,0.000000,0.000000,0.000000,0.000500,0.000500,0.000500,0.000500,0.000500,0.000261,0.000250,2
realized_pnl,593,"-14,179.039976","-9,096.529158","-3,079.877101",-496.183663,0.000000,0.000000,"1,314.478918","2,835.854261","22,825.975525",-404.034132,"2,053.646420",250


# 13. Data Integrity Audit

마지막으로 이후 Notebook을 실행해도 되는지를 PASS / WARN / FAIL 형태로 요약한다.

WARN은 데이터 오류를 의미하지 않는다.

예:

- derivatives state variation 부족
- 아직 maturity가 끝나지 않은 24h outcome
- expected feature가 candidate에서만 존재

등은 정상적인 연구 제약일 수 있다.

In [55]:
audit_results = []


def add_audit(name, status, detail):
    assert status in {
        "PASS",
        "WARN",
        "FAIL"
    }

    audit_results.append({
        "check": name,
        "status": status,
        "detail": detail,
    })

In [56]:
add_audit(
    "Snapshot ID uniqueness",
    "PASS" if snapshots["snapshot_id"].is_unique else "FAIL",
    f"{len(snapshots):,} rows"
)

add_audit(
    "Logical snapshot grain",
    "PASS" if logical_duplicate_count == 0 else "FAIL",
    f"{logical_duplicate_count:,} duplicates"
)

invalid_price_count = (
    snapshots["reference_price"]
    .dropna()
    .le(0)
    .sum()
)

add_audit(
    "Reference price validity",
    "PASS" if invalid_price_count == 0 else "FAIL",
    f"{invalid_price_count:,} <= 0"
)

invalid_liquidity_count = (
    snapshots["liquidity"]
    .dropna()
    .lt(0)
    .sum()
)

add_audit(
    "Liquidity validity",
    "PASS" if invalid_liquidity_count == 0 else "FAIL",
    f"{invalid_liquidity_count:,} < 0"
)

invalid_volatility_count = (
    snapshots["volatility"]
    .dropna()
    .lt(0)
    .sum()
)

add_audit(
    "Volatility validity",
    "PASS" if invalid_volatility_count == 0 else "FAIL",
    f"{invalid_volatility_count:,} < 0"
)

add_audit(
    "Reference timestamp",
    "PASS" if len(reference_after_decision) == 0 else "FAIL",
    f"{len(reference_after_decision):,} reference_at > decision_time"
)

In [57]:
numeric_snapshot = snapshots[
    FEATURE_COLUMNS
].apply(
    pd.to_numeric,
    errors="coerce"
)

inf_count = np.isinf(
    numeric_snapshot.to_numpy(dtype=float)
).sum()

add_audit(
    "Numeric infinity",
    "PASS" if inf_count == 0 else "FAIL",
    f"{inf_count:,} +/- inf values"
)

In [58]:
completed_with_null = int(
    outcome_integrity.iloc[0][
        "completed_with_null"
    ] or 0
)

missing_with_values = int(
    outcome_integrity.iloc[0][
        "missing_with_values"
    ] or 0
)

add_audit(
    "Outcome PK uniqueness",
    "PASS" if outcome_duplicates.empty else "FAIL",
    f"{len(outcome_duplicates):,} duplicates"
)

add_audit(
    "Completed outcome completeness",
    "PASS" if completed_with_null == 0 else "FAIL",
    f"{completed_with_null:,} invalid rows"
)

add_audit(
    "Missing outcome consistency",
    "PASS" if missing_with_values == 0 else "WARN",
    f"{missing_with_values:,} MISSING_DATA rows containing values"
)

add_audit(
    "MFE / MAE ordering",
    "PASS" if mfe_lt_mae == 0 else "FAIL",
    f"{mfe_lt_mae:,} MFE < MAE"
)

In [59]:
score_null_pct = (
    snapshots["score"].isna().mean() * 100
)

add_audit(
    "Score coverage",
    "WARN",
    (
        f"{score_null_pct:.2f}% NULL — expected because "
        "scores exist only for scoring candidates"
    )
)

In [60]:
basis_rate_coverage = 0

if "basisRate" in context_flat.columns:
    basis_rate_coverage = (
        context_flat["basisRate"]
        .notna()
        .mean()
        * 100
    )

add_audit(
    "Official basisRate coverage",
    "WARN",
    (
        f"{basis_rate_coverage:.2f}% non-null; "
        "use basisInputRate / markIndexBasisRate"
    )
)

if "state" in context_flat.columns:

    squeeze_states = (
        context_flat["state"]
        .dropna()
        .nunique()
    )

    add_audit(
        "Squeeze state variation",
        "PASS" if squeeze_states > 1 else "WARN",
        f"{squeeze_states} unique state(s)"
    )

if "crowdingState" in context_flat.columns:

    crowding_states = (
        context_flat["crowdingState"]
        .dropna()
        .nunique()
    )

    add_audit(
        "Crowding state variation",
        "PASS" if crowding_states > 1 else "WARN",
        f"{crowding_states} unique state(s)"
    )

In [61]:
add_audit(
    "V2.8 frozen feature cohort",
    "PASS" if frozen_feature_pass else "FAIL",
    (
        f"{int(frozen_consistency['mismatched_groups'].sum()):,} "
        "feature-group mismatches"
    )
)

In [62]:
audit_df = pd.DataFrame(audit_results)

status_order = {
    "FAIL": 0,
    "WARN": 1,
    "PASS": 2,
}

audit_df["_order"] = audit_df[
    "status"
].map(status_order)

audit_df = (
    audit_df
    .sort_values(
        ["_order", "check"]
    )
    .drop(columns="_order")
    .reset_index(drop=True)
)

display(audit_df)

,check,status,detail
0,V2.8 frozen feature cohort,FAIL,"5,980 feature-group mismatches"
1,Crowding state variation,WARN,1 unique state(s)
2,Official basisRate coverage,WARN,0.00% non-null; use basisInputRate / markIndex...
3,Score coverage,WARN,93.05% NULL — expected because scores exist on...
4,Squeeze state variation,WARN,1 unique state(s)
5,Completed outcome completeness,PASS,0 invalid rows
6,Liquidity validity,PASS,0 < 0
7,Logical snapshot grain,PASS,0 duplicates
8,MFE / MAE ordering,PASS,0 MFE < MAE
9,Missing outcome consistency,PASS,0 MISSING_DATA rows containing values


In [63]:
fail_count = (
    audit_df["status"] == "FAIL"
).sum()

warn_count = (
    audit_df["status"] == "WARN"
).sum()

pass_count = (
    audit_df["status"] == "PASS"
).sum()

print("=" * 70)
print("01 DATA AUDIT")
print("=" * 70)

print(f"PASS : {pass_count}")
print(f"WARN : {warn_count}")
print(f"FAIL : {fail_count}")

print("-" * 70)

if fail_count > 0:
    READINESS = "FAIL"
    print(
        "ANALYSIS READINESS: FAIL"
    )
    print(
        "Critical integrity problems must be investigated."
    )

else:
    READINESS = "PASS"
    print(
        "ANALYSIS READINESS: PASS"
    )
    print(
        "No critical integrity failure detected."
    )

print("=" * 70)

01 DATA AUDIT
PASS : 11
WARN : 4
FAIL : 1
----------------------------------------------------------------------
ANALYSIS READINESS: FAIL
Critical integrity problems must be investigated.


# Conclusion

이 Notebook의 PASS는 전략에 edge가 있다는 뜻이 아니다.

의미는 다음과 같다.

> 현재 V2.8 observation / outcome 데이터가 후속 research를 수행할 수 있을 정도로 구조적·시간적 무결성을 가지고 있다.

다음 Notebook:

`02_edge_baseline.ipynb`

에서는 처음으로 전략 성과를 분석한다.

핵심 질문:

1. 전체 market opportunity의 forward return은 얼마인가?
2. scoring candidate로 좁히면 기대수익률이 개선되는가?
3. selected candidate는 rejected candidate보다 실제로 우월한가?
4. gross expectancy가 양수인가?
5. 최소 round-trip cost 0.20%를 차감한 뒤에도 edge가 존재하는가?
6. V2.8 네 shadow lane 중 동일 cohort에서 어떤 selection 방식이 가장 나은가?

주의:

Forward Outcome에는 거래비용이 포함되어 있지 않으므로 후속 분석에서 비용을 별도로 적용한다.

```python
ROUND_TRIP_COST = 0.002


### 한 군데는 실행해보고 확인하세요

`Cell 51`의 frozen cohort 비교에서 제가 `decision_time + asset`을 cohort key로 사용했습니다. Codex 조사에서는 V2.8이 동일 시점/시장에서 shadow 실행됐고 snapshot 자체의 안전한 key는 `experiment_id + decision_id + asset`이라고 확인됐습니다. fileciteturn0file0L174-L193

만약 **4개 lane의 `decision_id`도 동일하게 공유된다는 게 확인되면**, Cell 51은 오히려 다음처럼 바꾸는 게 더 정확합니다.

```python
grouped = snapshots.groupby(
    ["decision_id", "asset"],
    sort=False
)